# Milestone 5 - DL&GenAI Project Course

Milestone Assignment Objective: The Messy Mashup Challenge
Your ultimate goal for this project is to build an end-to-end deep learning pipeline capable of classifying music genres in highly noisy environments. You will not just be writing snippets of code; you will be developing a complete system.

The questions below serve as your architectural blueprint. By answering them and verifying your outputs, you will incrementally construct a bug-free data generation process, a robust PyTorch DataLoader, and a state-of-the-art Audio Spectrogram Transformer (AST) model.

Your Final Deliverables:

Build & Verify: Use the questions below to construct and test your pipeline components.

Train: Once your architecture is verified, run your training loop! Let your model learn to separate the instrument stems from the environmental noise.

Predict & Submit: After training, you must write an inference script to process the unseen, unlabelled audio files listed in the test.csv file. Generate your predictions, format them into a submission.csv file, and upload your submission to the Kaggle competition leaderboard to report your final performance!

Treat these questions as milestones. If your code outputs the correct answers here, you can be confident that your foundation is solid before you spend hours training on the GPU. Good luck!

## Q1)
Dataset Splitting Strategy Task: You are building a lazy-loading "recipe" generator for the 10 genres.

Instead of loading audio into RAM, you will create a list of dictionaries. Assume you create exactly 100 recipe dictionaries per genre. Pass this combined list of recipes into sklearn.model_selection.train_test_split using test_size=0.2, shuffle=True, and random_state=42.

Question: Exactly how many recipe items will be allocated to your validation set (val_recipes) list?

20
200
100
1000

-> 20 per genere * 10 genres = 200

## Q2)
On-the-Fly Mixing Dimensions Task: In your training loop, you will load audio stems on the fly. 

Assume you use librosa.load(sr=16000, duration=10) to load 4 instrument stems and 1 noise file. You pad or truncate all 5 arrays to be exactly 160,000 samples long. 

You sum the 4 stems together to create a mix array, and then add the noise array scaled by an intensity weight of 0.2. 

Question: Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be? 


(160000,)
(160000,2)
(160000,5)
None Of The Above

-> (160000,) as no effect on shape of x

## Question 3: Hugging Face Feature Extractor Shape Task: 


Initialize Hugging Face's AutoFeatureExtractor using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension. 


Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)
*
2 points
[1024, 128]
[1024,128,10]
[1024,12,128]
None Of The Above

In [ ]:
import numpy as np
from transformers import AutoFeatureExtractor

# Load the feature extractor
extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

mix = np.ones(160000) # dummy audio

# Pass through extractor
output = extractor(mix, sampling_rate=16000, return_tensors="pt")

tensor = output["input_values"].squeeze(0) # Extract and squeeze

print(tensor.shape)  

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

c:\Users\dhruv\miniconda3\envs\messy_mashup\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhruv\.cache\huggingface\hub\models--MIT--ast-finetuned-audioset-10-10-0.4593. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


torch.Size([1024, 128])


## Q4:
Model Architecture Initialization Task: 
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

Question: Write a quick loop using 

sum(p.numel() for p in model.parameters() if p.requires_grad) 

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?


86196490
87406318
874318
87406718

In [3]:
from transformers import ASTForAudioClassification
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)
# Count trainable parameters
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


86196490


## Q5):
Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula: 

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?

-1.000
-2.000
+1.000
None Of the above

->
max(abs(y_test)) = 0.85

y_norm[0] = -0.85 / (0.85 + 1e-9) ≈ -1.000